
# 02 Préparation et fiabilisation des données

**Projet :** Life Expectancy (2000–2015)

**Objectif :** Préparer et fiabiliser le jeu de données à partir des problèmes identifiés lors du diagnostic initial. La démarche consiste à intégrer des données officielles lorsque cela est justifié, harmoniser les variables, traiter les anomalies et les valeurs manquantes, puis valider les transformations afin d'obtenir un jeu de données fiable et exploitable pour les analyses.

**Référence Agile :**

Epic 2 — Préparation globales des données (LEP-2)

Story LEP-13 — Préparer les données pour l’analyse

## Travaux à réaliser

- Conservation du jeu de données original et création d'une copie de travail.
- Acquisition des données officielles auprès de sources fiables.
- Harmonisation des pays, des années, des variables et des unités.
- Comparaison des données originales et des données officielles.
- Remplacement contrôlé des variables présentant des incohérences.
- Traitement des anomalies restantes.
- Validation de la qualité des données après préparation.
- Documentation des transformations et exportation du jeu de données fiabilisé.


## 2.1 Initialisation et conservation des données originales

### 2.1.1 Importation des bibliothèques

In [93]:

# Importer les bibliothèques nécessaires
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import shutil # Importer le module de copie de fichiers
import re

### 2.1.2 Définition des chemins et création des dossiers

In [ ]:
# Préparation des dossiers pour l'importation des données

# Définir le répertoire principal des données
DATA_DIR = Path("../data")

# Définir les sous-dossiers du projet
RAW_DIR = DATA_DIR / "raw" # pour la donnée de base
EXTERNAL_DIR = DATA_DIR / "external" # pour les sources externes
PROCESSED_DIR = DATA_DIR / "processed" # pour la donnée traitée

# Créer les sous-dossiers 
for dossier in [RAW_DIR, EXTERNAL_DIR, PROCESSED_DIR]:
    dossier.mkdir(parents=True, exist_ok=True)


### 2.1.3 Chargement des données originales et création d'une copie

In [4]:

# Charger le fichier original 
df_raw = pd.read_csv(DATA_DIR / "Life_Expectancy.csv")

# Créer une copie indépendante pour les transformations
df = df_raw.copy(deep=True)   # Copie profonde pour isoler les données de travail et préserver le DataFrame d'origine (df_raw)


In [5]:
# Sauvegarde d'une copie du fichier initiale dans le dossier raw

# Définir les chemins source et destination
source = DATA_DIR / "Life_Expectancy.csv"
destination = RAW_DIR / "Life_Expectancy.csv"

# Conserver une copie physique sans écraser une copie existante
if not destination.exists():
    shutil.copy2(source, destination)
    print("Copie originale conservée dans :", destination)
else:
    print("Une copie existe déjà dans :", destination)


Une copie existe déjà dans : ..\data\raw\Life_Expectancy.csv


### 2.2	Acquisition des données officielles

#### 2.2.1	Identification et documentation des indicateurs officiels


**Objectif :** Identifier les indicateurs officiels susceptibles de servir à la validation ou à la fiabilisation des variables problématiques du jeu de données WHO Life Expectancy (2000–2015).

Les indicateurs sont recherchés auprès de la Banque mondiale, de l'Organisation mondiale de la santé (OMS) et du Programme des Nations unies pour le développement (PNUD).

Pour chaque variable, le catalogue documente la source, le code de l'indicateur, l'unité, le lien de référence et le niveau de comparabilité à vérifier.

Les données officielles seront examinées avant toute transformation du jeu original.

#### 2.2.2	Configuration des API et des fonctions de téléchargement



Nous allons configurer l'accès aux sources officielles et développer une fonction réutilisable permettant de télécharger les séries historiques des indicateurs retenus pour la période 2000–2015.

L'acquisition débute par l'API Indicators V2 de la Banque mondiale. La fonction prend en charge les paramètres des requêtes, la pagination, la gestion des erreurs et la structuration des observations en DataFrame Pandas.

Les données de l'OMS et du PNUD feront l'objet de procédures d'acquisition adaptées à leurs services respectifs. Aucune transformation des données originales n'est effectuée à cette étape.

In [6]:

# Définir l'adresse de base de l'API Banque mondiale
BASE_URL_WB = "https://api.worldbank.org/v2"

# Définir les paramètres de téléchargement
PARAMS_WB = {
    "date": "2000:2015",
    "format": "json",
    "source": 2,               # Sélectionner World Development Indicators
    "per_page": 20000          # Demander jusqu'à 20 000 observations par page
}


In [86]:

# Définir la fonction de téléchargement
def telecharger_indicateur_wb(code, nom_variable):
    """
    Télécharge un indicateur de la Banque mondiale pour la période 2000–2015.

    Interroge l'API World Bank (source WDI), récupère les observations
    disponibles pour tous les pays et retourne un DataFrame structuré
    par pays et par année.

    Args:
        code (str):
            Code officiel de l'indicateur de la Banque mondiale.
            Exemple : "SP.POP.TOTL" pour la population totale.

        nom_variable (str):
            Nom de la colonne contenant les valeurs de l'indicateur
            dans le DataFrame retourné.
            Exemple : "Population_WB".

    Returns:
        pd.DataFrame:
            Tableau contenant les colonnes suivantes :
            - ISO3 : code ISO à trois lettres du pays.
            - Country_WB : nom du pays selon la Banque mondiale.
            - Year : année de l'observation (entier).
            - nom_variable : valeur numérique de l'indicateur.

            Les valeurs non numériques ou absentes sont converties
            en NaN.

    Raises:
        requests.exceptions.RequestException:
            Si la requête HTTP échoue (connexion, délai, statut HTTP).

        ValueError:
            Si l'API indique plusieurs pages de résultats,
            car la fonction ne gère pas la pagination.

    Example:
        >>> df_population = telecharger_indicateur_wb(
        ...     "SP.POP.TOTL",
        ...     "Population_WB"
        ... )
        >>> df_population.head()

    Notes:
        - La période étudiée est fixée à 2000–2015.
        - La source 2 correspond aux World Development Indicators.
        - La variable BASE_URL_WB doit être définie au préalable.
        - Les données manquantes sont conservées sous forme de NaN.
        - Les résultats incluent les entités agrégées présentes
          dans l'API, en plus des pays.
    """

    # Construire l'URL de l'indicateur
    url = f"{BASE_URL_WB}/country/all/indicator/{code}"

    # Définir les paramètres de la requête
    params = {
        "date": "2000:2015",
        "format": "json",
        "source": 2,
        "per_page": 10000
    }

    # Télécharger les données
    reponse = requests.get(
        url,
        params=params,
        timeout=700
    )
    reponse.raise_for_status()

    # Extraire les métadonnées et les observations
    meta, observations = reponse.json()

    # Vérifier que toutes les observations ont été reçues
    if int(meta["pages"]) != 1:
        raise ValueError(
            "Plusieurs pages disponibles : téléchargement incomplet."
        )

    # Construire le DataFrame
    donnees = pd.DataFrame([
        {
            "ISO3": obs["countryiso3code"],
            "Country_WB": obs["country"]["value"],
            "Year": int(obs["date"]),
            nom_variable: obs["value"]
        }
        for obs in observations
    ])

    # Convertir les valeurs en numérique
    donnees[nom_variable] = pd.to_numeric(
        donnees[nom_variable],
        errors="coerce"
    )

    return donnees


In [87]:
# documentation de la fonction telecharger_indicateur_wb()
help(telecharger_indicateur_wb)

Help on function telecharger_indicateur_wb in module __main__:

telecharger_indicateur_wb(code, nom_variable)
    Télécharge un indicateur de la Banque mondiale pour la période 2000–2015.
    
    Interroge l'API World Bank (source WDI), récupère les observations
    disponibles pour tous les pays et retourne un DataFrame structuré
    par pays et par année.
    
    Args:
        code (str):
            Code officiel de l'indicateur de la Banque mondiale.
            Exemple : "SP.POP.TOTL" pour la population totale.
    
        nom_variable (str):
            Nom de la colonne contenant les valeurs de l'indicateur
            dans le DataFrame retourné.
            Exemple : "Population_WB".
    
    Returns:
        pd.DataFrame:
            Tableau contenant les colonnes suivantes :
            - ISO3 : code ISO à trois lettres du pays.
            - Country_WB : nom du pays selon la Banque mondiale.
            - Year : année de l'observation (entier).
            - nom_variable 

In [ ]:

# Télécharger les données de la variable Population
population_wb = telecharger_indicateur_wb(
    "SP.POP.TOTL",
    "Population_WB"
)

# Vérifier les résultats
print("Dimensions :", population_wb.shape)
display(population_wb.head())


Dimensions : (4256, 4)


,ISO3,Country_WB,Year,Population_WB
0,AFE,Africa Eastern and Southern,2015,607123269.0
1,AFE,Africa Eastern and Southern,2014,590968990.0
2,AFE,Africa Eastern and Southern,2013,575202699.0
3,AFE,Africa Eastern and Southern,2012,559609961.0
4,AFE,Africa Eastern and Southern,2011,544737983.0


>Notre première acquisition des données officielles de la Banque mondiale a réussi. L'API a répondu et nous avons récupéré les observations de Population pour la période 2000–2015.

#### 2.2.3	Acquisition d'autres indicateurs Banque mondiale



>Ici, nous ferons un téléchargement csv directement car les l'API de la banque mondiale ne répondaient pas pour la sélection de prochaines variables. 

La fonction ci-dessous télécharge un ZIP, extrait le CSV des données, conserve les années 2000–2015 et retourne un DataFrame avec les colonnes dont nous avons besoin.

In [89]:

# Importer les bibliothèques nécessaires
from io import BytesIO
from zipfile import ZipFile

# Définir la fonction de téléchargement CSV
def telecharger_csv_wb(code, nom_variable):
    """
    Télécharge un indicateur de la Banque mondiale au format CSV.

    Récupère l'archive ZIP officielle de la base World Development
    Indicators (WDI), extrait le fichier CSV contenant les données,
    puis transforme les observations du format large au format long
    pour la période 2000–2015.

    Args:
        code (str):
            Code officiel de l'indicateur de la Banque mondiale.
            Exemple : "SP.POP.TOTL" pour la population totale.

        nom_variable (str):
            Nom de la colonne contenant les valeurs de l'indicateur
            dans le DataFrame retourné.
            Exemple : "Population_WB".

    Returns:
        pd.DataFrame:
            Tableau au format long contenant quatre colonnes :
            - ISO3 : code pays à trois caractères.
            - Country_WB : nom du pays selon la Banque mondiale.
            - Year : année de l'observation (entier).
            - nom_variable : valeur numérique de l'indicateur.

            Les valeurs absentes ou non numériques sont converties
            en NaN.

    Raises:
        requests.exceptions.RequestException:
            Si la requête HTTP échoue (connexion, délai, statut HTTP).

        zipfile.BadZipFile:
            Si le contenu téléchargé n'est pas une archive ZIP valide.

        ValueError:
            Si aucun fichier CSV de données commençant par "API_"
            n'est trouvé dans l'archive, ou si une conversion échoue.

        KeyError:
            Si les colonnes attendues (pays, code ou années)
            sont absentes du fichier CSV.

    Example:
        >>> df_population = telecharger_csv_wb(
        ...     "SP.POP.TOTL",
        ...     "Population_WB"
        ... )
        >>> df_population.head()

    Notes:
        - La source 2 correspond aux World Development Indicators.
        - La période étudiée est fixée à 2000–2015.
        - Le téléchargement utilise le format CSV compressé (ZIP).
        - Le fichier ZIP est traité directement en mémoire.
        - Les quatre premières lignes du CSV sont ignorées.
        - La méthode melt() transforme les années en observations.
        - Les données manquantes sont conservées sous forme de NaN.
        - Les entités agrégées peuvent être présentes dans les données.
        - Les colonnes ISO3 et Country_WB conservent les libellés
          fournis par la Banque mondiale.
    """

    # Construire l'URL du téléchargement
    url = (
        f"https://api.worldbank.org/v2/"
        f"country/all/indicator/{code}"
    )

    # Télécharger le fichier ZIP officiel
    reponse = requests.get(
        url,
        params={
            "source": 2,
            "downloadformat": "csv"
        },
        timeout=(15, 120)
    )

    reponse.raise_for_status()

    # Ouvrir le fichier ZIP en mémoire
    with ZipFile(BytesIO(reponse.content)) as archive:

        # Identifier le fichier CSV contenant les données
        fichiers = [
            nom for nom in archive.namelist()
            if nom.startswith("API_")
            and nom.endswith(".csv")
        ]

        if not fichiers:
            raise ValueError("CSV des données introuvable.")

        # Charger le CSV dans Pandas
        with archive.open(fichiers[0]) as fichier:
            donnees = pd.read_csv(
                fichier,
                skiprows=4
            )

    # Sélectionner les années 2000 à 2015
    annees = [str(an) for an in range(2000, 2016)]

    # Conserver uniquement les colonnes nécessaires
    donnees = donnees[
        ["Country Name", "Country Code"] + annees
    ]

    # Transformer les années en lignes
    donnees = donnees.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=annees,
        var_name="Year",
        value_name=nom_variable
    )

    # Renommer les colonnes
    donnees = donnees.rename(columns={
        "Country Name": "Country_WB",
        "Country Code": "ISO3"
    })

    # Convertir les types
    donnees["Year"] = donnees["Year"].astype(int)

    donnees[nom_variable] = pd.to_numeric(
        donnees[nom_variable],
        errors="coerce"
    )

    return donnees


In [90]:
# Consulter la documentation de la fonction
help(telecharger_csv_wb)

Help on function telecharger_csv_wb in module __main__:

telecharger_csv_wb(code, nom_variable)
    Télécharge un indicateur de la Banque mondiale au format CSV.
    
    Récupère l'archive ZIP officielle de la base World Development
    Indicators (WDI), extrait le fichier CSV contenant les données,
    puis transforme les observations du format large au format long
    pour la période 2000–2015.
    
    Args:
        code (str):
            Code officiel de l'indicateur de la Banque mondiale.
            Exemple : "SP.POP.TOTL" pour la population totale.
    
        nom_variable (str):
            Nom de la colonne contenant les valeurs de l'indicateur
            dans le DataFrame retourné.
            Exemple : "Population_WB".
    
    Returns:
        pd.DataFrame:
            Tableau au format long contenant quatre colonnes :
            - ISO3 : code pays à trois caractères.
            - Country_WB : nom du pays selon la Banque mondiale.
            - Year : année de l'obser

>Téléchargement du PIB par habitant "GDP_WB" ci-dessous avec la fonction.

In [91]:
# Télécharger le PIB par habitant dont le code est "NY.GDP.PCAP.CD"
gdp_wb = telecharger_csv_wb(
    "NY.GDP.PCAP.CD",
    "GDP_WB"
)

# Vérifier le résultat
print("Dimensions :", gdp_wb.shape)

display(gdp_wb.head())


Dimensions : (4256, 4)


,Country_WB,ISO3,Year,GDP_WB
0,Aruba,ABW,2000,20681.023027
1,Africa Eastern and Southern,AFE,2000,706.728441
2,Afghanistan,AFG,2000,174.930991
3,Africa Western and Central,AFW,2000,519.642600
4,Angola,AGO,2000,563.733796


Nous avons donc téléchargé la données sur le PIB par habitant sous forme de dataframe `gdp_wb`. Puisque Population et GDP ont déjà été téléchargés, nous les conservons en mémoire et nous récupérons uniquement les  indicateurs possibles restants.

### Téléchargement des indicateurs  

Nous cherchons à récupérer plusieurs indicateurs officiels de la Banque mondiale (*World Development Indicators — WDI*) pour la période **2000–2015**.

Les indicateurs ciblés  sont les suivants :

| Indicateur | Code WDI |
|---|---|
| Population totale | `SP.POP.TOTL` |
| PIB par habitant (USD courants) | `NY.GDP.PCAP.CD` |
| Vaccination contre l'hépatite B | `SH.IMM.HEPB` |
| Dépenses courantes de santé (% du PIB) | `SH.XPD.CHEX.GD.ZS` |
| Consommation d'alcool par habitant | `SH.ALC.PCAP.LI` |
| Vaccination DTC (DTP3) | `SH.IMM.IDPT` |
| Espérance de vie à la naissance | `SP.DYN.LE00.IN` |

#### Note de changement de méthode de téléchargement

Une première tentative de récupération des données a été réalisée à l'aide de l'API de la Banque mondiale, en utilisant des requêtes HTTP au format JSON.

Cependant, cette méthode a rencontré plusieurs erreurs de serveur et de délai de réponse (notamment **502, 504 et 524**), empêchant un téléchargement fiable des indicateurs.

Nous allons donc tester une **méthode alternative**, qui consiste à télécharger directement les fichiers CSV officiels compressés (ZIP), puis à les extraire et à les transformer avec Python et Pandas.

Voici les liens :
- 
**Sources officielles — Banque mondiale :**

- [Catalogue officiel — World Development Indicators (WDI)](https://datacatalog.worldbank.org/search/dataset/0037712/world-development-indicators)
- [Téléchargement direct — Base WDI au format ZIP](https://databank.worldbank.org/data/download/WDI_CSV.zip)


Cette approche vise à améliorer la fiabilité du téléchargement tout en conservant la traçabilité des sources et la reproductibilité du traitement.

Le fichier téléchargé manuellement est WDI_CSV_2026_10_08.zip et mis dans notre dossier External.

In [11]:
# On charge le nouveau fichier téléchargé et vérifier son contenu

# Importer la bibliothèque de lecture des archives
from zipfile import ZipFile

# Définir le chemin du fichier téléchargé
fichier_wdi = EXTERNAL_DIR / "WDI_CSV_2026_10_08.zip"

# Vérifier la présence du fichier
print("Fichier disponible :", fichier_wdi.exists())

# Examiner le contenu de l'archive
with ZipFile(fichier_wdi) as archive:
    for fichier in archive.namelist():
        print(fichier)


Fichier disponible : True
WDICSV.csv
WDICountry.csv
WDISeries.csv
WDIcountry-series.csv
WDIfootnote.csv
WDIseries-time.csv


Résultat : True, suivi de la liste des fichiers présents dans l'archive.

Nous recherchons notamment WDIData.csv, qui contient les observations des indicateurs par pays et année

In [12]:
# Extraire les indicateurs ciblés dans un dataframe wdi_selection

## ce code parcourt le fichier WDI par blocs de 10 000 lignes, 
# conserve uniquement les sept indicateurs recherchés et 
# rassemble les résultats dans un DataFrame appelé wdi_selection.

# Définir les indicateurs officiels à extraire
INDICATEURS_WB = {
    "Population_WB": "SP.POP.TOTL",
    "GDP_WB": "NY.GDP.PCAP.CD",
    "Hepatitis_B_WB": "SH.IMM.HEPB",
    "Health_Expenditure_WB": "SH.XPD.CHEX.GD.ZS",
    "Alcohol_WB": "SH.ALC.PCAP.LI",
    "Diphtheria_WB": "SH.IMM.IDPT",
    "Life_Expectancy_WB": "SP.DYN.LE00.IN"
}

# Définir les années de notre étude
annees = [str(an) for an in range(2000, 2016)]



# Définir les colonnes nécessaires
colonnes = [
    "Country Name",
    "Country Code",
    "Indicator Code"
] + annees

# Initialiser la liste des observations sélectionnées
extraits = []

# Ouvrir l'archive WDI
with ZipFile(fichier_wdi) as archive:

    # Ouvrir le fichier principal des données
    with archive.open("WDICSV.csv") as fichier:

        # Lire les données par blocs
        for bloc in pd.read_csv(
            fichier,
            usecols=colonnes,
            chunksize=10000
        ):

            # Sélectionner uniquement les indicateurs recherchés
            selection = bloc[
                bloc["Indicator Code"].isin(
                    INDICATEURS_WB.values()
                )
            ]

            # Conserver les lignes sélectionnées
            if not selection.empty:
                extraits.append(selection)

# Regrouper les observations extraites
wdi_selection = pd.concat(
    extraits,
    ignore_index=True
) if extraits else pd.DataFrame(columns=colonnes)

# Vérifier les dimensions
print("Dimensions :", wdi_selection.shape)


Dimensions : (1862, 19)


Nous vérifierons ensuite que les 7 indicateurs sont réellement présents dans wdi_selection.

In [13]:

# Identifier les codes effectivement présents
codes_trouves = set(wdi_selection["Indicator Code"])

# Vérifier la présence de chaque indicateur
for nom_variable, code in INDICATEURS_WB.items():

    statut = (
        "Présent"
        if code in codes_trouves
        else "Absent"
    )

    print(f"{nom_variable} ({code}) : {statut}")


Population_WB (SP.POP.TOTL) : Présent
GDP_WB (NY.GDP.PCAP.CD) : Présent
Hepatitis_B_WB (SH.IMM.HEPB) : Présent
Health_Expenditure_WB (SH.XPD.CHEX.GD.ZS) : Présent
Alcohol_WB (SH.ALC.PCAP.LI) : Présent
Diphtheria_WB (SH.IMM.IDPT) : Présent
Life_Expectancy_WB (SP.DYN.LE00.IN) : Présent


Constat : Les sept indicateurs recherchés sont présents dans wdi_selection.

> Ensuite nous allons contrôler les données officielles acquises, Examiner la couverture de chaque indicateur

> Créer le DataFrame wdi_long.


Notre fichier WDI contient initialement une colonne par année alors que notre projet utilise une structure dans laquelle chaque ligne correspond à un pays et une année. Nous alllons donc transformer wdi_selection du format large en format long avec melt(). Ce nouveau format où chaque ligne correspondra à une entité, une année et un indicateur.

In [92]:

# Définir les années de la période d'étude
annees = [str(an) for an in range(2000, 2016)]

# Transformer les années en lignes
wdi_long = wdi_selection.melt(
    id_vars=[
        "Country Name",
        "Country Code",
        "Indicator Code"
    ],
    value_vars=annees,
    var_name="Year",
    value_name="Valeur"
)

# Renommer les colonnes
wdi_long = wdi_long.rename(columns={
    "Country Name": "Country_WB",
    "Country Code": "ISO3"
})

# Convertir les types de données
wdi_long["Year"] = wdi_long["Year"].astype(int)

wdi_long["Valeur"] = pd.to_numeric(
    wdi_long["Valeur"],
    errors="coerce"
)

# Vérifier le résultat
print("Dimensions :", wdi_long.shape)
display(wdi_long.head())


Dimensions : (29792, 5)


,Country_WB,ISO3,Indicator Code,Year,Valeur
0,Africa Eastern and Southern,AFE,SH.XPD.CHEX.GD.ZS,2000,5.657523
1,Africa Eastern and Southern,AFE,NY.GDP.PCAP.CD,2000,706.728441
2,Africa Eastern and Southern,AFE,SH.IMM.IDPT,2000,55.525288
3,Africa Eastern and Southern,AFE,SH.IMM.HEPB,2000,7.773944
4,Africa Eastern and Southern,AFE,SP.DYN.LE00.IN,2000,52.557342


Nous allons sélectionner les pays de notre jeu Kaggle, puis effectuer une jointure pour comparer les données.

In [ ]:
# Sélectionner les pays de Kaggle

# Extraire les pays présents dans le jeu original
pays_kaggle = df_raw["Country"].str.strip().unique()

# Vérifier le nombre de pays
print("Nombre de pays Kaggle :", len(pays_kaggle))


Nombre de pays Kaggle : 193


In [ ]:
# Sélectionner les pays correspondants dans WDI, soit wdi_long

# Créer une copie des données officielles
wdi_selection_pays = wdi_long.copy()

# Normaliser les noms des pays
wdi_selection_pays["Country_key"] = (
    wdi_selection_pays["Country_WB"]
    .str.strip()
    .str.casefold()
)

# Normaliser les noms Kaggle
pays_kaggle_key = (
    pd.Series(pays_kaggle)
    .str.casefold()
)

# Sélectionner les pays correspondants
wdi_selection_pays = wdi_selection_pays[
    wdi_selection_pays["Country_key"].isin(pays_kaggle_key)
].copy()

print("Pays correspondants :",
      wdi_selection_pays["Country_WB"].nunique())


Pays correspondants : 163


Constat : Nous avons obtenu 163 pays correspondants entre notre jeu de données Kaggle et les données officielles de la Banque mondiale.

Cela signifie que 163 pays ont exactement le même nom dans les deux sources, après normalisation des majuscules et des espaces.

La prochaine étape consiste à identifier les pays de Kaggle qui n'ont pas encore de correspondance, avant d'effectuer la jointure.

In [ ]:
# Ce code identifie les pays non correspondants

# Identifier les pays présents dans Kaggle
pays_kaggle = set(
    df_raw["Country"].str.strip().str.casefold()
)

# Identifier les pays correspondants dans WDI
pays_wdi = set(wdi_selection_pays["Country_key"])

# Rechercher les pays sans correspondance
pays_non_correspondants = sorted(
    pays_kaggle - pays_wdi
)

# Afficher les résultats
print("Pays Kaggle :", len(pays_kaggle))
print("Pays correspondants :", len(pays_kaggle & pays_wdi))
print("Pays sans correspondance :", len(pays_non_correspondants))

display(pd.DataFrame({
    "Pays_Kaggle": pays_non_correspondants
}))


Pays Kaggle : 193
Pays correspondants : 163
Pays sans correspondance : 30


,Pays_Kaggle
0,bahamas
1,bolivia (plurinational state of)
2,congo
3,cook islands
4,côte d'ivoire
5,democratic people's republic of korea
6,democratic republic of the congo
7,egypt
8,gambia
9,iran (islamic republic of)


Constat : Nous avons maintenant identifié les 30 pays de Kaggle qui ne correspondent pas directement aux noms utilisés par la Banque mondiale.

La plupart de ces différences concernent simplement la dénomination officielle des pays. Nous allons donc procéder en deux étapes simples :
1. Associer les 30 noms Kaggle à leurs codes ISO3, sans modifier df_raw.
2. Vérifier les codes présents dans wdi_long, puis effectuer la jointure sur ISO3 et Year.

>Harmonisation des noms de pays et attribution des codes ISO3 : Afin de faciliter la jointure entre le jeu de données initial *Life Expectancy* et les indicateurs de la Banque mondiale (WDI), nous utilisons les codes pays à trois caractères fournis dans le fichier officiel WDI. 
Les correspondances sont établies directement à partir des colonnes `Country Name` et `Country Code` du référentiel WDI. 
Pour les pays dont les appellations diffèrent entre les deux sources, une table de correspondance explicite est construite et vérifiée afin de garantir l'identification correcte des pays.


In [ ]:
# Ici nous définissons les correspondances des 30 pays avec leurs codes ISO

# Définir les codes ISO3 des pays sans correspondance directe
correspondances = {
    "bahamas": "BHS",
    "bolivia (plurinational state of)": "BOL",
    "congo": "COG",
    "cook islands": "COK",
    "côte d'ivoire": "CIV",
    "democratic people's republic of korea": "PRK",
    "democratic republic of the congo": "COD",
    "egypt": "EGY",
    "gambia": "GMB",
    "iran (islamic republic of)": "IRN",
    "kyrgyzstan": "KGZ",
    "lao people's democratic republic": "LAO",
    "micronesia (federated states of)": "FSM",
    "nauru": "NRU",
    "niue": "NIU",
    "republic of korea": "KOR",
    "republic of moldova": "MDA",
    "saint kitts and nevis": "KNA",
    "saint lucia": "LCA",
    "saint vincent and the grenadines": "VCT",
    "slovakia": "SVK",
    "somalia": "SOM",
    "swaziland": "SWZ",
    "the former yugoslav republic of macedonia": "MKD",
    "turkey": "TUR",
    "united kingdom of great britain and northern ireland": "GBR",
    "united republic of tanzania": "TZA",
    "united states of america": "USA",
    "venezuela (bolivarian republic of)": "VEN",
    "yemen": "YEM"
}


In [ ]:
# Ensuite, nous vérifions la présence de ces codes dans wdi_long

# Récupérer les codes géographiques disponibles dans WDI
codes_wdi = set(wdi_long["ISO3"].dropna().unique())

# Vérifier les correspondances proposées
verification = pd.DataFrame([
    {
        "Pays_Kaggle": pays,
        "ISO3": code,
        "Present_WDI": code in codes_wdi
    }
    for pays, code in correspondances.items()
])

# Afficher les résultats
display(verification)

# Afficher les codes absents
display(verification[~verification["Present_WDI"]])


,Pays_Kaggle,ISO3,Present_WDI
0,bahamas,BHS,True
1,bolivia (plurinational state of),BOL,True
2,congo,COG,True
3,cook islands,COK,False
4,côte d'ivoire,CIV,True
5,democratic people's republic of korea,PRK,True
6,democratic republic of the congo,COD,True
7,egypt,EGY,True
8,gambia,GMB,True
9,iran (islamic republic of),IRN,True


,Pays_Kaggle,ISO3,Present_WDI
3,cook islands,COK,False
14,niue,NIU,False


In [ ]:
## Ensuite compléter automatiquement les codes des 163 pays déjà correspondants
# Nous pouvons récupérer leurs codes directement depuis wdi_long, puisque leurs noms correspondent.

# Construire une table des pays et codes WDI
table_wdi = (
    wdi_long[["Country_WB", "ISO3"]]
    .drop_duplicates()
    .copy()
)

# Normaliser les noms
table_wdi["Country_key"] = (
    table_wdi["Country_WB"]
    .str.strip()
    .str.casefold()
)

# Construire une table des pays Kaggle
table_pays = pd.DataFrame({
    "Country": df_raw["Country"].drop_duplicates()
})

table_pays["Country_key"] = (
    table_pays["Country"]
    .str.strip()
    .str.casefold()
)

# Récupérer les ISO3 par correspondance directe
table_pays = table_pays.merge(
    table_wdi[["Country_key", "ISO3"]],
    on="Country_key",
    how="left",
    validate="one_to_one"
)

# Compléter les pays ayant des noms différents
table_pays["ISO3"] = table_pays["ISO3"].fillna(
    table_pays["Country_key"].map(correspondances)
)

# Vérifier les résultats
print("Pays Kaggle :", len(table_pays))
print("Codes ISO3 attribués :", table_pays["ISO3"].notna().sum())

display(table_pays[table_pays["ISO3"].isna()])


Pays Kaggle : 193
Codes ISO3 attribués : 193


,Country,Country_key,ISO3


Résultat : Nous avons maintenant attribué un code ISO3 aux 193 pays du jeu de données Kaggle dans notre table de correspondances table_pays.

Ensuite nous allons utiliser cette table de correspondance pour ajouter les codes ISO3 à une copie de Kaggle (df_comparaison) ci-dessous.

In [26]:

# Créer une copie du jeu de données original
df_comparaison = df_raw.copy()

# Ajouter les codes ISO3 à partir de la table de correspondance
df_comparaison = df_comparaison.merge(
    table_pays[["Country", "ISO3"]],
    on="Country",
    how="left",
    validate="many_to_one"
)

# Vérifier le résultat
print("Dimensions :", df_comparaison.shape)
print("Codes ISO3 manquants :", df_comparaison["ISO3"].isna().sum())


Dimensions : (2938, 23)
Codes ISO3 manquants : 0


In [ ]:
## Préparation des indicateurs WDI pour la jointure : À partir 
# de wdi_long, nous conservons seulement les codes ISO3 des pays Kaggle.


# Sélectionner les pays présents dans Kaggle
wdi_pays = wdi_long[
    wdi_long["ISO3"].isin(table_pays["ISO3"])
].copy()

# Vérifier l'unicité des observations
doublons = wdi_pays.duplicated(
    ["ISO3", "Year", "Indicator Code"]
).sum()

print("Doublons pays-année-indicateur :", doublons)
print("Pays WDI correspondants :", wdi_pays["ISO3"].nunique())


Doublons pays-année-indicateur : 0
Pays WDI correspondants : 191


Constat : On constate que 2 pays sont absentx. Nous allons les identifier ci-dessous.

In [28]:

# Identifier les codes ISO3 présents dans WDI
codes_wdi = set(wdi_pays["ISO3"].dropna())

# Rechercher les pays Kaggle absents de WDI
pays_absents = table_pays[
    ~table_pays["ISO3"].isin(codes_wdi)
]

# Afficher les pays concernés
display(pays_absents[["Country", "ISO3"]])


,Country,ISO3
39,Cook Islands,COK
124,Niue,NIU


> Jointure des données Kaggle avec les indicateurs officiels WDI

>Afin de comparer les variables du jeu initial avec les indicateurs officiels de la Banque mondiale, nous procédons à deux jointures successives.

>La première jointure associe à chaque pays du jeu Kaggle son code ISO3 à partir de la table de correspondance précédemment constituée, en utilisant `Country` comme clé.

>La seconde jointure enrichit les observations Kaggle avec les indicateurs WDI correspondants, en utilisant la clé composite `ISO3`–`Year`.

>Les deux jointures sont réalisées à gauche (`left join`) afin de conserver l'ensemble des observations du jeu initial. 

In [ ]:
## Dans ce code, nous réalisons deux jointures successives, avec deux objectifs différents :
# 1. Première jointure : ajouter les codes ISO3 aux pays du jeu Kaggle.
# 2. Deuxième jointure : enrichir le jeu Kaggle avec les indicateurs officiels de la Banque mondiale (WDI).

# Préparer les données officielles au format large
wdi_pays_wide = wdi_pays.pivot(
    index=["ISO3", "Year"],
    columns="Indicator Code",
    values="Valeur"
).reset_index()

# Renommer les indicateurs
wdi_pays_wide = wdi_pays_wide.rename(columns={
    code: nom
    for nom, code in INDICATEURS_WB.items()
})

wdi_pays_wide.columns.name = None

# Créer une nouvelle copie du jeu Kaggle
df_comparaison = df_raw.copy()

# Ajouter les codes ISO3
df_comparaison = df_comparaison.merge(
    table_pays[["Country", "ISO3"]],
    on="Country",
    how="left",
    validate="many_to_one"
)

# Ajouter les indicateurs officiels
df_comparaison = df_comparaison.merge(
    wdi_pays_wide,
    on=["ISO3", "Year"],
    how="left",
    validate="many_to_one",
    indicator=True
)

# Vérifier les résultats
print("Dimensions originales :", df_raw.shape)
print("Dimensions après jointure :", df_comparaison.shape)

print("\nRésultats de la jointure :")
print(df_comparaison["_merge"].value_counts())


Dimensions originales : (2938, 22)
Dimensions après jointure : (2938, 31)

Résultats de la jointure :
_merge
both          2936
left_only        2
right_only       0
Name: count, dtype: int64


In [ ]:

# Vérifier les deux observations non correspondantes : examiner les observations sans correspondance WDI
display(
    df_comparaison.loc[
        df_comparaison["_merge"] == "left_only",
        ["Country", "ISO3", "Year"]
    ]
)


,Country,ISO3,Year
624,Cook Islands,COK,2013
1909,Niue,NIU,2013


Les deux observations sans correspondance concernent Cook Islands (COK) et Niue (NIU), toutes deux pour l'année 2013.

Interprétation : Ces deux territoires ne sont pas représentés dans l'extraction WDI utilisée. La couverture des observations est néanmoins très élevée.

> Ensuite faisons un controle de comparaison des valeurs manquantes entre Kaggle et WDI

In [ ]:
# Supprimer les espaces au début et à la fin des noms des colonnes
df_comparaison.columns = df_comparaison.columns.str.strip()

# Vérifier les colonnes après harmonisation
print(df_comparaison.columns.tolist())


['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality', 'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling', 'ISO3', 'GDP_WB', 'Alcohol_WB', 'Hepatitis_B_WB', 'Diphtheria_WB', 'Health_Expenditure_WB', 'Life_Expectancy_WB', 'Population_WB', '_merge']


In [ ]:
# ce code pépare un tableau synthese comparaison
# Associer les variables Kaggle aux indicateurs officiels
variables_comparaison = {
    "Population": "Population_WB",
    "GDP": "GDP_WB",
    "Hepatitis B": "Hepatitis_B_WB",
    "Total expenditure": "Health_Expenditure_WB",
    "Alcohol": "Alcohol_WB",
    "Diphtheria": "Diphtheria_WB",
    "Life expectancy": "Life_Expectancy_WB"
}

# Initialiser les résultats
resultats = []

# Comparer la disponibilité des données
for var_kaggle, var_wb in variables_comparaison.items():

    manquantes_kaggle = df_comparaison[var_kaggle].isna()
    disponibles_wb = df_comparaison[var_wb].notna()

    recuperables = (
        manquantes_kaggle & disponibles_wb
    ).sum()

    resultats.append({
        "Variable": var_kaggle,
        "Manquantes_Kaggle": manquantes_kaggle.sum(),
        "Disponibles_WDI": disponibles_wb.sum(),
        "Potentiellement_recuperables": recuperables
    })

# Construire le tableau de synthèse
synthese_comparaison = pd.DataFrame(resultats)

display(synthese_comparaison)


,Variable,Manquantes_Kaggle,Disponibles_WDI,Potentiellement_recuperables
0,Population,652,2936,650
1,GDP,448,2908,418
2,Hepatitis B,553,2377,1
3,Total expenditure,226,2846,180
4,Alcohol,194,2888,176
5,Diphtheria,19,2917,0
6,Life expectancy,10,2936,8


Ce tableau ci-dessus permet de mesurer concrètement l'apport potentiel des données officielles de la Banque mondiale aux valeurs manquantes de Kaggle.

In [ ]:

# Identifier les observations manquantes dans Kaggle pour hetatis
hepatitis_manquantes = df_comparaison[
    df_comparaison["Hepatitis B"].isna()
].copy()

# Résumer la disponibilité WDI par année
couverture_hepatitis = (
    hepatitis_manquantes
    .groupby("Year")
    .agg(
        Manquantes_Kaggle=("Country", "size"),
        Disponibles_WDI=("Hepatitis_B_WB", "count")
    )
    .reset_index()
)

# Afficher le tableau
display(couverture_hepatitis)


,Year,Manquantes_Kaggle,Disponibles_WDI
0,2000,98,0
1,2001,88,0
2,2002,70,0
3,2003,52,0
4,2004,45,1
5,2005,36,0
6,2006,32,0
7,2007,24,0
8,2008,20,0
9,2009,17,0


In [ ]:

# Identifier les pays les plus concernés par les valeurs manquantes de Hepatitis B dans Kaggle
pays_hepatitis_manquants = (
    df_comparaison[
        df_comparaison["Hepatitis B"].isna()
    ]
    .groupby("Country")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="Annees_manquantes")
)

# Afficher les pays concernés
display(pays_hepatitis_manquants.head(20))


,Country,Annees_manquantes
0,Finland,16
1,Denmark,16
2,Norway,16
3,Slovenia,16
4,United Kingdom of Great Britain and Northern I...,16
5,Switzerland,16
6,Japan,16
7,Hungary,16
8,Iceland,16
9,South Sudan,15


Constat : L'analyse des valeurs manquantes de Hepatitis B révèle une concentration temporelle au début de la période 2000–2015 et une forte hétérogénéité entre les pays. Neuf pays présentés dans le tableau disposent d'une série entièrement manquante sur les 16 années. La Banque mondiale ne permet de récupérer qu'une seule des 553 valeurs absentes de Kaggle.

Interprétation : Les données manquantes présentent une structure temporelle et géographique marquée. Les lacunes ne peuvent être traitées par une simple imputation uniforme, particulièrement pour les pays dont toute la série est absente.

Décision : Conserver provisoirement les données originales, rechercher une source complémentaire OMS/UNICEF et évaluer sa couverture avant de décider d'un éventuel remplacement.

In [36]:
df_comparaison.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Schooling,ISO3,GDP_WB,Alcohol_WB,Hepatitis_B_WB,Diphtheria_WB,Health_Expenditure_WB,Life_Expectancy_WB,Population_WB,_merge
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,10.1,AFG,565.569730,0.009,64.0,64.0,10.105348,62.270,33831764.0,both
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,10.0,AFG,625.054942,0.004,62.0,62.0,9.528878,62.260,32792523.0,both
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,9.9,AFG,637.087099,0.004,64.0,64.0,8.805964,62.188,31622704.0,both
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,9.8,AFG,651.417134,0.004,67.0,67.0,7.897169,61.735,30560034.0,both
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,9.5,AFG,606.694676,0.008,68.0,68.0,8.561908,61.250,29347708.0,both


Passage à la création de df_clean, notre nouveau DataFrame destiné à l'étude.

In [37]:
# Créer une copie indépendante
df_clean = df_comparaison.copy()

# Harmoniser les noms des colonnes
df_clean.columns = df_clean.columns.str.strip()

In [ ]:
# Utiliser les données officielles pour Population et GDP en remplacant les valeurs lorsque WDI est disponible
df_clean["Population"] = df_clean["Population_WB"].combine_first(
    df_clean["Population"]
)

df_clean["GDP"] = df_clean["GDP_WB"].combine_first(
    df_clean["GDP"]
)

In [ ]:
# Vérifier les valeurs manquantes apress remplacement
manquants = df_clean[df_raw.columns.str.strip()].isna().sum()

display(manquants[manquants > 0].sort_values(ascending=False))

Hepatitis B                        553
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
BMI                                 34
thinness  1-19 years                34
thinness 5-9 years                  34
GDP                                 30
Polio                               19
Diphtheria                          19
Adult Mortality                     10
Life expectancy                     10
Population                           2
dtype: int64

Donc après l'intégration des données WDI, Population et GDP sont désormais presque complets. Passons au traitement des autres variables manquantes (il s'agit notemment des variables sanitaires : Hepatitis B, Polio, Diphtheria, BMI, thinness 1-19 years et thinness 5-9 years )



#### Consultation des données officielles de vaccination OMS/UNICEF (WUENIC)

Après l'intégration des indicateurs de la Banque mondiale (WDI), nous examinons les **estimations OMS/UNICEF de la couverture vaccinale nationale (WUENIC)** afin de compléter et de fiabiliser les données relatives à la vaccination.

Pour cela, nous procédons comme suit :

1. Accéder au site [UNICEF DATA — Immunization](https://data.unicef.org/topic/child-health/immunization/).
2. Sélectionner **Immunization coverage by antigen (country, regional, and global trends)**, mise à jour du 14 juillet 2026.
3. Télécharger le fichier regroupant les estimations de couverture vaccinale par antigène.
4. Enregistrer le fichier wuenic2025rev_web-update dans le répertoire `data/external/` du projet.

Cette base fournit des estimations officielles de la couverture vaccinale par pays, par année et par antigène. Elle permettra notamment d'examiner les indicateurs relatifs à l'hépatite B (`HepB3`), à la diphtérie-tétanos-coqueluche (`DTP3`) et à la poliomyélite (`Pol3`).

**Source :** [UNICEF DATA — Immunization](https://data.unicef.org/topic/child-health/immunization/)


In [40]:

# Identifier les fichiers WUENIC téléchargés
fichiers_wuenic = list(
    EXTERNAL_DIR.glob("*wuenic2025rev_web-update*")
)

# Afficher les fichiers trouvés
for fichier in fichiers_wuenic:
    print(fichier.name, "| Extension :", fichier.suffix)


wuenic2025rev_web-update.xlsx | Extension : .xlsx


In [41]:
# Examiner les feuilles du fichier WUENIC


# Définir le chemin du fichier WUENIC
fichier_wuenic = (
    EXTERNAL_DIR / "wuenic2025rev_web-update.xlsx"
)

# Ouvrir le classeur Excel
classeur_wuenic = pd.ExcelFile(fichier_wuenic)

# Afficher les feuilles disponibles
print("Feuilles disponibles :")
print(classeur_wuenic.sheet_names)


Feuilles disponibles :
['country', 'region']


In [ ]:
# Nous allons examiner la structure de la feuille country

# Charger la feuille contenant les données par pays
df_wuenic = pd.read_excel(
    fichier_wuenic,
    sheet_name="country"
)

# Vérifier les dimensions
print("Dimensions :", df_wuenic.shape)

# Afficher les noms des colonnes
print("Colonnes :", df_wuenic.columns.tolist())

# Examiner les premières observations
display(df_wuenic.head())


Dimensions : (58291, 15)
Colonnes : ['iso3c', 'country', 'vaccine', 'year', 'target_grp', 'coverage', 'source', 'unicef', 'gavi', 'world_bank', 'world_bank_fcv', 'who', 'who_fcv', 'african_union', 'european_union']


,iso3c,country,vaccine,year,target_grp,coverage,source,unicef,gavi,world_bank,world_bank_fcv,who,who_fcv,african_union,european_union
0,AFG,Afghanistan,bcg,1982,births,10,WHO/UNICEF Estimates of National Immunization ...,ROSA,Gavi57,low income,wb_fcv,emr,fcv,NaN,NaN
1,AFG,Afghanistan,bcg,1983,births,10,WHO/UNICEF Estimates of National Immunization ...,ROSA,Gavi57,low income,wb_fcv,emr,fcv,NaN,NaN
2,AFG,Afghanistan,bcg,1984,births,11,WHO/UNICEF Estimates of National Immunization ...,ROSA,Gavi57,low income,wb_fcv,emr,fcv,NaN,NaN
3,AFG,Afghanistan,bcg,1985,births,17,WHO/UNICEF Estimates of National Immunization ...,ROSA,Gavi57,low income,wb_fcv,emr,fcv,NaN,NaN
4,AFG,Afghanistan,bcg,1986,births,18,WHO/UNICEF Estimates of National Immunization ...,ROSA,Gavi57,low income,wb_fcv,emr,fcv,NaN,NaN


In [ ]:

# Vérifions les codes exacts des trois vaccins Hepatitis B, Polio et Diphtheria .


# Identifier les codes des vaccins recherchés
vaccins = df_wuenic[
    df_wuenic["vaccine"].astype(str).str.upper().isin(
        ["HEPB3", "POL3", "DTP3"]
    )
]

# Vérifier les vaccins trouvés
print(vaccins["vaccine"].value_counts())


vaccine
dtp3     8340
hepb3    4944
Name: count, dtype: int64


Le résultat confirme que nous avons retrouvé deux vaccins dans le fichier WUENIC : Diphtérie, tétanos, coqueluche	(dtp3) et Hépatite B (3 doses)	hepb3.



In [44]:

# Rechercher les codes vaccinaux liés à la poliomyélite
codes_polio = (
    df_wuenic.loc[
        df_wuenic["vaccine"].str.contains(
            "pol", case=False, na=False
        ),
        "vaccine"
    ]
    .value_counts()
)

display(codes_polio)


Series([], Name: count, dtype: int64)

Le résultat confirme qu'aucun code contenant pol n'est présent dans ton fichier wuenic2025rev_web-update.xlsx.

Nous pouvons donc poursuivre avec les deux indicateurs disponibles :
- hepb3 → Hepatitis B
- dtp3 → Diphtheria


Pour Polio, nous conserverons provisoirement les données initiale déjà présentes dans df_comparaison.

In [45]:
# Avant la jointure, vérifions la couverture réelle des deux vaccins sur notre période 2000–2015.

# Sélectionner les deux vaccins et la période d'étude
wuenic_selection = df_wuenic[
    df_wuenic["vaccine"].str.lower().isin(["hepb3", "dtp3"])
    & df_wuenic["year"].between(2000, 2015)
].copy()

# Vérifier la disponibilité des valeurs
print(
    wuenic_selection.groupby("vaccine")["coverage"]
    .agg(["size", "count"])
)

# Contrôler les doublons
print(
    "Doublons pays-année-vaccin :",
    wuenic_selection.duplicated(
        ["iso3c", "year", "vaccine"]
    ).sum()
)


         size  count
vaccine             
dtp3     3101   3101
hepb3    2555   2555
Doublons pays-année-vaccin : 0


**Constat :** Pour la période 2000–2015, les données OMS/UNICEF sélectionnées contiennent 3 101 observations pour le vaccin DTP3 et 2 555 pour le vaccin HepB3. Aucune valeur manquante n'est détectée parmi les observations présentes et aucun doublon n'est identifié sur la clé composite pays–année–vaccin.

In [ ]:

# Mettre les codes vaccinaux en colonnes
wuenic_wide = wuenic_selection.pivot(
    index=["iso3c", "year"],
    columns="vaccine",
    values="coverage"
).reset_index()

# Renommer les colonnes
wuenic_wide = wuenic_wide.rename(columns={
    "iso3c": "ISO3",
    "year": "Year",
    "hepb3": "Hepatitis_B_WUENIC",
    "dtp3": "Diphtheria_WUENIC"
})

wuenic_wide.columns.name = None

display(wuenic_wide.head())


,ISO3,Year,Diphtheria_WUENIC,Hepatitis_B_WUENIC
0,AFG,2000,24.0,NaN
1,AFG,2001,33.0,NaN
2,AFG,2002,36.0,NaN
3,AFG,2003,41.0,NaN
4,AFG,2004,50.0,NaN


In [ ]:
# Jointure avec df_clean : # Passons maintenant à la jointure avec df_clean, sans modifier ses valeurs originales.


# Ajouter les données WUENIC au jeu de travail
df_clean = df_clean.merge(
    wuenic_wide,
    on=["ISO3", "Year"],
    how="left",
    validate="many_to_one"
)

# Vérifier les dimensions
print("Dimensions :", df_clean.shape)

Dimensions : (2938, 33)


In [49]:
# Comparer les valeurs existantes des deux sources
comparaison = df_clean[
    ["Hepatitis B", "Hepatitis_B_WUENIC",
     "Diphtheria", "Diphtheria_WUENIC"]
]

# Calculer les corrélations entre sources
print(comparaison.corr())

                    Hepatitis B  Hepatitis_B_WUENIC  Diphtheria  \
Hepatitis B            1.000000            0.709217    0.611495   
Hepatitis_B_WUENIC     0.709217            1.000000    0.449680   
Diphtheria             0.611495            0.449680    1.000000   
Diphtheria_WUENIC      0.518162            0.713374    0.717535   

                    Diphtheria_WUENIC  
Hepatitis B                  0.518162  
Hepatitis_B_WUENIC           0.713374  
Diphtheria                   0.717535  
Diphtheria_WUENIC            1.000000  


In [50]:
#Vérifions les écarts entre les deux sources


# Comparer les écarts absolus entre Kaggle et WUENIC
for original, officiel in {
    "Hepatitis B": "Hepatitis_B_WUENIC",
    "Diphtheria": "Diphtheria_WUENIC"
}.items():

    # Conserver uniquement les observations communes
    comparaison = df_clean[[original, officiel]].dropna()

    # Calculer les écarts absolus
    ecarts = (
        comparaison[original] - comparaison[officiel]
    ).abs()

    print(f"\n{original}")
    print("Observations communes :", len(comparaison))
    print("Écart moyen absolu :", round(ecarts.mean(), 2))
    print("Écart médian absolu :", round(ecarts.median(), 2))
    print("Écart maximal :", round(ecarts.max(), 2))



Hepatitis B
Observations communes : 2378
Écart moyen absolu : 5.26
Écart médian absolu : 0.0
Écart maximal : 90.0

Diphtheria
Observations communes : 2919
Écart moyen absolu : 4.73
Écart médian absolu : 0.0
Écart maximal : 90.0


In [ ]:
# Remplacer les deux variables Hepatitis B et Diphtheria par les séries officielles

# Remplacer les deux variables par les séries officielles WUENIC
df_clean["Hepatitis B"] = df_clean["Hepatitis_B_WUENIC"]
df_clean["Diphtheria"] = df_clean["Diphtheria_WUENIC"]

# Vérifier les valeurs manquantes après remplacement
print(
    df_clean[["Hepatitis B", "Diphtheria"]]
    .isna()
    .sum()
)


Hepatitis B    559
Diphtheria      19
dtype: int64


In [ ]:
#Vérifier les conséquences du remplacement
#comparons les valeurs manquantes avant et après :


# Comparer les valeurs manquantes avant et après
comparaison_manquants = pd.DataFrame({
    "Kaggle_avant": [
        df_raw["Hepatitis B"].isna().sum(),
        df_raw["Diphtheria "].isna().sum()
    ],
    "WUENIC_apres": [
        df_clean["Hepatitis B"].isna().sum(),
        df_clean["Diphtheria"].isna().sum()
    ]
}, index=["Hepatitis B", "Diphtheria"])

display(comparaison_manquants)


,Kaggle_avant,WUENIC_apres
Hepatitis B,553,559
Diphtheria,19,19


In [ ]:
# Vérifier les dimensions et les valeurs manquantes
print("Dimensions :", df_clean.shape)

manquants = df_clean[
    df_raw.columns.str.strip()
].isna().sum()

display(manquants[manquants > 0].sort_values(ascending=False))

Dimensions : (2938, 33)


Hepatitis B                        559
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
BMI                                 34
thinness  1-19 years                34
thinness 5-9 years                  34
GDP                                 30
Polio                               19
Diphtheria                          19
Adult Mortality                     10
Life expectancy                     10
Population                           2
dtype: int64

In [54]:
# Enregistrer le DataFrame

# Définir le chemin de sauvegarde
fichier_sauvegarde = PROCESSED_DIR / "life_expectancy_avant_imputation.csv"

# Enregistrer le DataFrame sans son index
df_clean.to_csv(
    fichier_sauvegarde,
    index=False,
    encoding="utf-8-sig"
)

# Vérifier la sauvegarde
print("Fichier enregistré :", fichier_sauvegarde.exists())
print("Dimensions :", df_clean.shape)


Fichier enregistré : True
Dimensions : (2938, 33)


In [55]:
# Vérifier que le fichier est réutilisable

# Recharger le fichier sauvegardé
df_verification = pd.read_csv(fichier_sauvegarde)

# Vérifier les dimensions
print("Dimensions identiques :", df_verification.shape == df_clean.shape)


Dimensions identiques : True


Décision méthodologique


Ce fichier *life_expectancy_avant_imputation.csv* constitue notre point de sauvegarde avant traitement des valeurs manquantes.

Il contient   actuellement les données Kaggle enrichies et les remplacements officiels déjà effectués, avec leurs valeurs manquantes restantes.


### Etape finale Finalisation du jeu fiabilisé : vérifier les dernières incohérences et retenir les variables pertinentes

1. Vérification de percentage expenditure

Commençons par cette variable avant de finaliser notre jeu de données.

Objectif : déterminer si percentage expenditure est suffisamment fiable et pertinente pour être conservée dans notre étude sur l'espérance de vie.

Nous allons examiner sa distribution et ses valeurs atypiques, puis décider de son sort.

In [56]:

# Examiner la distribution de percentage expenditure
df_clean["percentage expenditure"].describe()


count     2938.000000
mean       738.251295
std       1987.914858
min          0.000000
25%          4.685343
50%         64.912906
75%        441.534144
max      19479.911610
Name: percentage expenditure, dtype: float64

In [57]:

# Afficher les 10 observations les plus élevées
display(
    df_clean[
        ["Country", "Year", "percentage expenditure", "GDP"]
    ].nlargest(10, "percentage expenditure")
)


,Country,Year,percentage expenditure,GDP
2522,Switzerland,2014,19479.91161,90511.648456
2523,Switzerland,2013,19099.04506,88851.237450
1545,Luxembourg,2008,18961.34860,120422.137934
2525,Switzerland,2011,18822.86732,91910.396457
2524,Switzerland,2012,18379.32974,87269.559478
1542,Luxembourg,2011,17028.52798,119025.057203
1539,Luxembourg,2014,16255.16198,123678.706919
1540,Luxembourg,2013,15515.75234,120000.140730
1547,Luxembourg,2006,15345.49070,90788.816427
1915,Norway,2010,15268.06445,90020.342258


Nous constatons que :
- Les valeurs dépassent largement 100, ce qui ne correspond pas à une mesure ordinaire exprimée en pourcentage.
- Les valeurs les plus élevées concernent des économies à PIB par habitant élevé.
- L'unité et le mode de calcul de la variable demeurent incertains.



La documentation Kaggle décrit percentage expenditure comme une dépense de santé rapportée au PIB par habitant, exprimée en pourcentage. Cette définition reste ambiguë et ne permet pas d'expliquer clairement les valeurs observées. 


Par ailleurs, les dépenses de santé en pourcentage du PIB constituent un indicateur distinct, dont la définition est précisément établie par l'OMS. 

Ces valeurs pourraient correspondre à des montants monétaires, mais elles paraissent élevées même pour des dépenses de santé par habitant.

In [ ]:

# Rechercher l'indicateur des dépenses de santé par habitant sur Banque mondiale
code = "SH.XPD.CHEX.PC.CD"

with ZipFile(fichier_wdi) as archive:
    with archive.open("WDICSV.csv") as fichier:

        depenses_wb = pd.concat(
            (
                bloc[bloc["Indicator Code"] == code]
                for bloc in pd.read_csv(
                    fichier,
                    chunksize=10000
                )
            ),
            ignore_index=True
        )

# Vérifier sa présence
print("Observations trouvées :", len(depenses_wb))


Observations trouvées : 266


In [ ]:
# Comparer avec percentage expenditure
# Nous allons transformer cet indicateur au format long, puis le joindre à notre jeu de données.
# Définir les années de l'étude
annees = [str(an) for an in range(2000, 2016)]

# Transformer les années en lignes
depenses_wb_long = depenses_wb.melt(
    id_vars=["Country Code"],
    value_vars=annees,
    var_name="Year",
    value_name="Health_Expenditure_PC_WB"
)

# Harmoniser les colonnes
depenses_wb_long = depenses_wb_long.rename(
    columns={"Country Code": "ISO3"}
)

depenses_wb_long["Year"] = (
    depenses_wb_long["Year"].astype(int)
)

# Joindre les valeurs officielles aux données Kaggle
df_test_depenses = df_clean.merge(
    depenses_wb_long,
    on=["ISO3", "Year"],
    how="left",
    validate="many_to_one"
)

# Comparer les valeurs des deux sources
display(
    df_test_depenses[
        ["Country", "Year",
         "percentage expenditure",
         "Health_Expenditure_PC_WB"]
    ].head(10)
)


,Country,Year,percentage expenditure,Health_Expenditure_PC_WB
0,Afghanistan,2015,71.279624,59.919605
1,Afghanistan,2014,73.523582,60.049511
2,Afghanistan,2013,73.219243,56.160435
3,Afghanistan,2012,78.184215,52.452473
4,Afghanistan,2011,7.097109,52.187218
5,Afghanistan,2010,79.679367,46.424389
6,Afghanistan,2009,56.762217,43.133518
7,Afghanistan,2008,25.873925,39.886250
8,Afghanistan,2007,10.910156,32.708931
9,Afghanistan,2006,17.171518,28.941263


Puisque notre objectif est de produire un dataset fiabilisé, nous allons remplacer cette variable ambiguë par l'indicateur officiel des dépenses courantes de santé par habitant, mais sous un nouveau nom clairement identifié.

In [60]:

# Partir du jeu enrichi avec les dépenses officielles
df_final = df_test_depenses.copy()

# Exclure l'indicateur original ambigu
df_final = df_final.drop(
    columns=["percentage expenditure"]
)

# Renommer l'indicateur officiel
df_final = df_final.rename(columns={
    "Health_Expenditure_PC_WB": "Health_Expenditure_PC_USD"
})

# Vérifier la nouvelle variable
display(
    df_final[
        ["Country", "Year", "Health_Expenditure_PC_USD"]
    ].head()
)


,Country,Year,Health_Expenditure_PC_USD
0,Afghanistan,2015,59.919605
1,Afghanistan,2014,60.049511
2,Afghanistan,2013,56.160435
3,Afghanistan,2012,52.452473
4,Afghanistan,2011,52.187218


In [61]:
# Avant de valider cette nouvelle variable, vérifions sa couverture.
# Vérifier les valeurs manquantes de l'indicateur officiel
print(
    "Valeurs manquantes :",
    df_final["Health_Expenditure_PC_USD"].isna().sum()
)

print(
    "Taux de valeurs manquantes (%) :",
    round(
        df_final["Health_Expenditure_PC_USD"].isna().mean() * 100,
        2
    )
)

# Examiner les statistiques descriptives
display(df_final["Health_Expenditure_PC_USD"].describe())

Valeurs manquantes : 92
Taux de valeurs manquantes (%) : 3.13


count    2846.000000
mean      802.095323
std      1485.975237
min         4.175786
25%        53.271414
50%       196.636818
75%       682.410110
max      9469.664062
Name: Health_Expenditure_PC_USD, dtype: float64

Constat : L'indicateur officiel des dépenses courantes de santé par habitant présente 2 846 observations disponibles sur 2 938, soit une couverture de 96,87 %. Sa distribution est fortement asymétrique à droite.

Cet indicateur possède une définition et une unité explicites, contrairement à percentage expenditure. Il constitue une variable exploitable pour les analyses économiques et sanitaires du projet.

Nous allons remplacer la variable ambiguë par Health_Expenditure_PC_USD, sans prétendre à une équivalence stricte.

2. Vérification de la variable BMI.

Nous devons déterminer si les valeurs de cette variable correspondent réellement à un indice de masse corporelle ou à un autre indicateur nutritionnel.

Commençons par examiner sa distribution et ses valeurs extrêmes :

In [62]:
# Commençons par examiner sa distribution et ses valeurs extrêmes : Objectif : identifier les valeurs suspectes avant de décider de conserver, corriger ou exclure cette variable.
# Examiner les statistiques descriptives de BMI
display(df_final["BMI"].describe())

# Examiner les valeurs les plus faibles
display(
    df_final[["Country", "Year", "BMI"]]
    .nsmallest(10, "BMI")
)

# Examiner les valeurs les plus élevées
display(
    df_final[["Country", "Year", "BMI"]]
    .nlargest(10, "BMI")
)

count    2904.000000
mean       38.321247
std        20.044034
min         1.000000
25%        19.300000
50%        43.500000
75%        56.200000
max        87.300000
Name: BMI, dtype: float64

,Country,Year,BMI
2887,Viet Nam,2002,1.0
207,Bangladesh,2000,1.4
2886,Viet Nam,2003,1.4
206,Bangladesh,2001,1.8
2885,Viet Nam,2004,1.9
1555,Madagascar,2014,2.0
283,Benin,2004,2.1
600,Comoros,2007,2.1
724,Democratic Republic of the Congo,2012,2.1
1024,Ghana,2001,2.1


,Country,Year,BMI
1812,Nauru,2013,87.3
1958,Palau,2013,83.3
624,Cook Islands,2013,82.8
1650,Marshall Islands,2013,81.6
2713,Tuvalu,2013,79.3
1378,Kiribati,2015,77.6
1909,Niue,2013,77.3
1379,Kiribati,2014,77.1
1380,Kiribati,2013,76.7
1381,Kiribati,2012,76.2


Ces résultats révèlent une incohérence importante entre le nom de la variable et les valeurs observées. Deux exemples attirent particulièrement l'attention :
- Viet Nam, 2002 : 1,0
- Nauru, 2013 : 87,3

Ces valeurs ne sont pas cohérentes avec une interprétation de BMI comme IMC moyen d'une population adulte, généralement exprimé en kg/m².

En revanche, des valeurs allant de 1 à 87,3 pourraient correspondre à des pourcentages, par exemple une mesure de prévalence du surpoids.

L'OMS distingue clairement :
- IMC moyen : mesure exprimée en kg/m².
- Prévalence du surpoids : pourcentage d'adultes dont l'IMC est supérieur ou égal à 25 kg/m². 
 
Les valeurs de notre variable Kaggle semblent davantage compatibles avec la deuxième définition.

D'ailleurs, la Banque mondiale propose un indicateur officiel correspondant : SH.STA.OWAD.ZS — Prevalence of overweight (% of adults).

In [ ]:
# Vérifier la présence de l'indicateur du surpoids dans les données banques mondiales téléchargées
# Rechercher l'indicateur de prévalence du surpoids
code_bmi = "SH.STA.OWAD.ZS"

# Vérifier sa présence dans les métadonnées WDI
with ZipFile(fichier_wdi) as archive:
    with archive.open("WDISeries.csv") as fichier:
        series_wdi = pd.read_csv(
            fichier,
            usecols=["Series Code", "Indicator Name"]
        )

# Afficher l'indicateur recherché
display(
    series_wdi[
        series_wdi["Series Code"] == code_bmi
    ]
)


,Series Code,Indicator Name


Rien a été trouvé pour SH.STA.OWAD.ZS. Nous allons rechercher du coté de l'OMS.


2.2. Acquisition et validation d'un indicateur officiel de surpoids de l'OMS

Afin d'examiner et de fiabiliser la variable `BMI` du jeu initial *Life Expectancy*, nous consultons les données officielles de l'**Organisation mondiale de la Santé (OMS)**, disponibles dans le *Global Health Observatory (GHO)*.

L'indicateur retenu concerne la **prévalence du surpoids chez les adultes (IMC ≥ 25 kg/m²)**, exprimée en pourcentage de la population adulte.

Le fichier a été téléchargé manuellement depuis le portail officiel de l'OMS :

- **Source :** [OMS — Global Health Observatory : Body Mass Index among Adults](https://www.who.int/data/gho/data/themes/topics/indicator-groups/GHO/bmi-among-adults)
- **Indicateur :** Overweight among adults, BMI ≥ 25, prevalence (%)
- **Format téléchargé :** Excel (`.xlsx`)
- **Nom du fichier :** `data.xlsx`
- **Emplacement dans le projet :** `data/external/data.xlsx`

Nous procéderons à l'examen de la structure du fichier, de la couverture temporelle et géographique, des valeurs manquantes et des éventuels doublons avant d'envisager son intégration au jeu de données.




In [ ]:

# Charger et examiner le fichier

# Définir le chemin du fichier OMS
fichier_bmi_oms = EXTERNAL_DIR / "data.xlsx"

# Vérifier sa présence
print("Fichier disponible :", fichier_bmi_oms.exists())

# Identifier les feuilles Excel
classeur = pd.ExcelFile(fichier_bmi_oms)
print("Feuilles :", classeur.sheet_names)

# Charger la première feuille
df_bmi_oms = pd.read_excel(fichier_bmi_oms)

# Examiner la structure
print("Dimensions :", df_bmi_oms.shape)
print("Colonnes :", df_bmi_oms.columns.tolist())

# Afficher les premières observations
display(df_bmi_oms.head())


Fichier disponible : True
Feuilles : ['Sheet1']
Dimensions : (26867, 34)
Colonnes : ['Applied filters:\nLocation type is Country\nDim1 type is Sex\nLANGUAGE_CODE is en\nIndicatorCode is NCD_BMI_25A', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33']


,Applied filters:\nLocation type is Country\nDim1 type is Sex\nLANGUAGE_CODE is en\nIndicatorCode is NCD_BMI_25A,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
2,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,26.48,NaN,41.03,33.5 [26.5-41.0],NaN,NaN,EN,2026-05-27 00:00:00
3,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,15.29,NaN,33.29,23.6 [15.3-33.3],NaN,NaN,EN,2026-05-27 00:00:00
4,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,32.32,NaN,53.93,42.8 [32.3-53.9],NaN,NaN,EN,2026-05-27 00:00:00


In [66]:

# Lire les premières lignes sans définir d'en-tête
apercu_oms = pd.read_excel(
    fichier_bmi_oms,
    sheet_name="Sheet1",
    header=None,
    nrows=12
)

# Afficher les premières lignes
display(apercu_oms)


,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,Applied filters:\nLocation type is Country\nDi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
3,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,26.48,NaN,41.03,33.5 [26.5-41.0],NaN,NaN,EN,2026-05-27 00:00:00
4,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,15.29,NaN,33.29,23.6 [15.3-33.3],NaN,NaN,EN,2026-05-27 00:00:00
5,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,32.32,NaN,53.93,42.8 [32.3-53.9],NaN,NaN,EN,2026-05-27 00:00:00
6,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2023,...,NaN,NaN,26.13,NaN,40,32.9 [26.1-40.0],NaN,NaN,EN,2026-05-27 00:00:00
7,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2023,...,NaN,NaN,15.2,NaN,32.4,23.1 [15.2-32.4],NaN,NaN,EN,2026-05-27 00:00:00
8,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2023,...,NaN,NaN,31.89,NaN,52.64,42.1 [31.9-52.6],NaN,NaN,EN,2026-05-27 00:00:00
9,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2022,...,NaN,NaN,25.8,NaN,39.1,32.3 [25.8-39.1],NaN,NaN,EN,2026-05-27 00:00:00


In [67]:

# Charger le fichier avec les véritables en-têtes
df_bmi_oms = pd.read_excel(
    fichier_bmi_oms,
    sheet_name="Sheet1",
    header=2
)

# Vérifier la structure
print("Dimensions :", df_bmi_oms.shape)
print("Colonnes :", df_bmi_oms.columns.tolist())

# Afficher les premières observations
display(df_bmi_oms.head())


Dimensions : (26865, 34)
Colonnes : ['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type', 'SpatialDimValueCode', 'Location', 'Period type', 'Period', 'IsLatestYear', 'Dim1 type', 'Dim1', 'Dim1ValueCode', 'Dim2 type', 'Dim2', 'Dim2ValueCode', 'Dim3 type', 'Dim3', 'Dim3ValueCode', 'DataSourceDimValueCode', 'DataSource', 'FactValueNumericPrefix', 'FactValueNumeric', 'FactValueUoM', 'FactValueNumericLowPrefix', 'FactValueNumericLow', 'FactValueNumericHighPrefix', 'FactValueNumericHigh', 'Value', 'FactValueTranslationID', 'FactComments', 'Language', 'DateModified']


,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
0,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,26.48,NaN,41.03,33.5 [26.5-41.0],NaN,NaN,EN,2026-05-27
1,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,15.29,NaN,33.29,23.6 [15.3-33.3],NaN,NaN,EN,2026-05-27
2,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2024,...,NaN,NaN,32.32,NaN,53.93,42.8 [32.3-53.9],NaN,NaN,EN,2026-05-27
3,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2023,...,NaN,NaN,26.13,NaN,40.00,32.9 [26.1-40.0],NaN,NaN,EN,2026-05-27
4,NCD_BMI_25A,"Overweight among adults, BMI ≥ 25 kg/m2 (age-s...",numeric,AFR,Africa,Country,AGO,Angola,Year,2023,...,NaN,NaN,15.20,NaN,32.40,23.1 [15.2-32.4],NaN,NaN,EN,2026-05-27


In [68]:

# Afficher les noms des colonnes avec leur position
for i, colonne in enumerate(df_bmi_oms.columns):
    print(i, ":", colonne)


0 : IndicatorCode
1 : Indicator
2 : ValueType
3 : ParentLocationCode
4 : ParentLocation
5 : Location type
6 : SpatialDimValueCode
7 : Location
8 : Period type
9 : Period
10 : IsLatestYear
11 : Dim1 type
12 : Dim1
13 : Dim1ValueCode
14 : Dim2 type
15 : Dim2
16 : Dim2ValueCode
17 : Dim3 type
18 : Dim3
19 : Dim3ValueCode
20 : DataSourceDimValueCode
21 : DataSource
22 : FactValueNumericPrefix
23 : FactValueNumeric
24 : FactValueUoM
25 : FactValueNumericLowPrefix
26 : FactValueNumericLow
27 : FactValueNumericHighPrefix
28 : FactValueNumericHigh
29 : Value
30 : FactValueTranslationID
31 : FactComments
32 : Language
33 : DateModified


In [69]:
#Vérifier les catégories de sexe
#Avant de sélectionner les observations, vérifions les catégories présentes dans le fichier.

# Identifier les catégories de sexe disponibles
display(
    df_bmi_oms[
        ["Dim1", "Dim1ValueCode"]
    ].drop_duplicates()
)

# Vérifier la période couverte
print(
    "Période :",
    df_bmi_oms["Period"].min(),
    "-",
    df_bmi_oms["Period"].max()
)


,Dim1,Dim1ValueCode
0,Both sexes,SEX_BTSX
1,Male,SEX_MLE
2,Female,SEX_FMLE


Période : 1980 - 2024


C'est `SEX_BTSX` qui nous interesse.

In [70]:
# Sélectionner les données OMS pour 2000–2015
# Avant la jointure avec Kaggle, vérifions également que les autres dimensions ne créent pas plusieurs observations par pays et par année.


# Examiner les dimensions supplémentaires
print("Dim2 :", df_bmi_oms["Dim2"].dropna().unique())
print("Dim3 :", df_bmi_oms["Dim3"].dropna().unique())


Dim2 : ['18+  years']
Dim3 : []


In [71]:

# Sélectionner les observations correspondant à notre étude
bmi_oms_selection = df_bmi_oms.loc[
    (df_bmi_oms["Dim1ValueCode"] == "SEX_BTSX")
    & (df_bmi_oms["Period"].between(2000, 2015)),
    ["SpatialDimValueCode", "Period", "FactValueNumeric"]
].copy()

# Renommer les colonnes
bmi_oms_selection = bmi_oms_selection.rename(columns={
    "SpatialDimValueCode": "ISO3",
    "Period": "Year",
    "FactValueNumeric": "Overweight_Adults_Pct_OMS"
})

# Vérifier les observations
print("Dimensions :", bmi_oms_selection.shape)
print("Pays :", bmi_oms_selection["ISO3"].nunique())
print("Années :", bmi_oms_selection["Year"].nunique())
print("Valeurs manquantes :",
      bmi_oms_selection["Overweight_Adults_Pct_OMS"].isna().sum())
print("Doublons pays-année :",
      bmi_oms_selection.duplicated(["ISO3", "Year"]).sum())

display(bmi_oms_selection.head())


Dimensions : (3184, 3)
Pays : 199
Années : 16
Valeurs manquantes : 0
Doublons pays-année : 0


,ISO3,Year,Overweight_Adults_Pct_OMS
27,AGO,2015,28.20
30,AGO,2014,27.65
33,AGO,2013,27.11
36,AGO,2012,26.59
39,AGO,2011,26.07


In [72]:
#Jointure avec notre jeu de données
#Nous allons ajouter l'indicateur officiel sans modifier BMI.


# Ajouter l'indicateur officiel OMS
df_final = df_final.merge(
    bmi_oms_selection,
    on=["ISO3", "Year"],
    how="left",
    validate="many_to_one"
)

# Vérifier les dimensions et la couverture
print("Dimensions :", df_final.shape)

print(
    "Valeurs OMS disponibles :",
    df_final["Overweight_Adults_Pct_OMS"].notna().sum()
)

print(
    "Valeurs OMS manquantes :",
    df_final["Overweight_Adults_Pct_OMS"].isna().sum()
)



Dimensions : (2938, 34)
Valeurs OMS disponibles : 2936
Valeurs OMS manquantes : 2


In [73]:
# Comparer BMI de Kaggle avec l'indicateur OMS

# Conserver les observations communes
comparaison_bmi = df_final[
    ["BMI", "Overweight_Adults_Pct_OMS"]
].dropna()

# Calculer la corrélation et les écarts absolus
ecarts = (
    comparaison_bmi["BMI"]
    - comparaison_bmi["Overweight_Adults_Pct_OMS"]
).abs()

print("Observations communes :", len(comparaison_bmi))
print("Corrélation :", round(
    comparaison_bmi.corr().iloc[0, 1], 3
))
print("Écart moyen absolu :", round(ecarts.mean(), 2))
print("Écart médian absolu :", round(ecarts.median(), 2))
print("Écart maximal :", round(ecarts.max(), 2))


Observations communes : 2904
Corrélation : 0.729
Écart moyen absolu : 8.97
Écart médian absolu : 5.21
Écart maximal : 79.4


Ces résultats confirment que BMI de Kaggle et l'indicateur officiel OMS ne sont pas équivalents, même s'ils présentent une relation positive. Nous allons donc conserver l'indicateur officiel OMS et exclure BMI de notre jeu fiabilisé.

Décision : Exclure la variable originale BMI et conserver l'indicateur officiel Overweight_Adults_Pct_OMS, dont la définition est clairement documentée. Aucune imputation n'est effectuée à cette étape.

In [74]:

# Exclure la variable BMI originale
df_final = df_final.drop(columns=["BMI"])

# Vérifier la couverture du nouvel indicateur
print(
    "Valeurs manquantes OMS :",
    df_final["Overweight_Adults_Pct_OMS"].isna().sum()
)

# Vérifier les dimensions
print("Dimensions :", df_final.shape)


Valeurs manquantes OMS : 2
Dimensions : (2938, 33)


3. Vérification des variables thinness.

Nous avons deux variables à examiner :
- thinness  1-19 years
- thinness 5-9 years


L'objectif est de vérifier leur cohérence et de déterminer si nous pouvons les conserver ou s'il faut rechercher des indicateurs officiels mieux définis.

In [75]:
#Examiner leurs distributions


# Définir les deux variables nutritionnelles
variables_thinness = [
    "thinness  1-19 years",
    "thinness 5-9 years"
]

# Examiner leurs statistiques descriptives
display(df_final[variables_thinness].describe().T)


,count,mean,std,min,25%,50%,75%,max
thinness 1-19 years,2904.0,4.839704,4.420195,0.1,1.6,3.3,7.2,27.7
thinness 5-9 years,2904.0,4.870317,4.508882,0.1,1.5,3.3,7.2,28.6


In [76]:

# Examiner les valeurs les plus élevées pour chaque variable
for variable in variables_thinness:

    print(f"\nVariable : {variable}")

    display(
        df_final[
            ["Country", "Year", variable]
        ].nlargest(10, variable)
    )



Variable : thinness  1-19 years


,Country,Year,thinness 1-19 years
1201,India,2000,27.7
1200,India,2001,27.5
1199,India,2002,27.4
1198,India,2003,27.3
1196,India,2005,27.2
1197,India,2004,27.2
1194,India,2007,27.1
1195,India,2006,27.1
1191,India,2010,27.0
1192,India,2009,27.0



Variable : thinness 5-9 years


,Country,Year,thinness 5-9 years
1201,India,2000,28.6
1200,India,2001,28.5
1199,India,2002,28.4
1198,India,2003,28.3
1197,India,2004,28.2
1196,India,2005,28.1
1194,India,2007,28.0
1195,India,2006,28.0
1193,India,2008,27.9
1191,India,2010,27.8


Les résultats sont plutôt cohérents avec des indicateurs de prévalence de la maigreur exprimés en pourcentage. Contrairement à BMI, nous n'observons pas ici de valeurs manifestement incompatibles avec cette interprétation.

Point à clarifier
La variable thinness  1-19 years pose une difficulté : les indicateurs nutritionnels internationaux utilisent couramment la tranche d'âge 5–19 ans, plutôt que 1–19 ans.

Il est donc possible que le libellé du fichier Kaggle soit incorrect, mais nous ne devons pas le corriger sans vérification.

In [77]:
#Vérification complémentaire simple
#Avant une éventuelle acquisition, examinons la relation entre nos deux variables :

# Comparer les deux indicateurs de maigreur
print(
    "Corrélation :",
    round(
        df_final[variables_thinness].corr().iloc[0, 1],
        3
    )
)

# Calculer les différences absolues entre les deux variables
ecarts_thinness = (
    df_final[variables_thinness[0]]
    - df_final[variables_thinness[1]]
).abs()

print("Écart moyen absolu :", round(ecarts_thinness.mean(), 2))
print("Écart maximal :", round(ecarts_thinness.max(), 2))


Corrélation : 0.939
Écart moyen absolu : 0.33
Écart maximal : 18.6


Constat : Les variables thinness  1-19 years et thinness 5-9 years présentent une corrélation très élevée (r = 0,939) et un écart moyen absolu de 0,33 point. Toutefois, l'écart maximal atteint 18,6 points.

Interprétation : Les deux indicateurs présentent des évolutions généralement proches. Les écarts observés montrent néanmoins qu'ils ne sont pas strictement identiques.

Décision : Conserver provisoirement les deux variables, sans imputation. Leur définition exacte sera documentée.

4. Vérification des variables infant deaths et under-five deaths.

D'après notre diagnostic initial, infant deaths et under-five deaths ne présentent aucune valeur manquante.

Décision : Conserver les deux variables sans transformation dans le jeu fiabilisé. 


## 5. Validation du jeu de données fiabilisé, avant son exportation définitive.

In [79]:
# Vérification des dimensions et de la structure

# Comparer les dimensions initiales et finales
print("Dimensions originales :", df_raw.shape)
print("Dimensions finales :", df_final.shape)

# Vérifier les types des variables
df_final.info()


Dimensions originales : (2938, 22)
Dimensions finales : (2938, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   Country                          2938 non-null   object  
 1   Year                             2938 non-null   int64   
 2   Status                           2938 non-null   object  
 3   Life expectancy                  2928 non-null   float64 
 4   Adult Mortality                  2928 non-null   float64 
 5   infant deaths                    2938 non-null   int64   
 6   Alcohol                          2744 non-null   float64 
 7   Hepatitis B                      2379 non-null   float64 
 8   Measles                          2938 non-null   int64   
 9   under-five deaths                2938 non-null   int64   
 10  Polio                            2919 non-null   float64 
 11  To

Constats
- Toutes les 2 938 observations originales ont été conservées.
- Les variables BMI et percentage expenditure ont été retirées.
- Deux variables officielles ont été ajoutées : Health_Expenditure_PC_USD et Overweight_Adults_Pct_OMS.
- Les valeurs manquantes ont été conservées volontairement.
- Les 11 autres colonnes supplémentaires comprennent les indicateurs de comparaison et les colonnes techniques.
Nous pouvons donc revenir à 22 variables d'étude, comme dans le jeu original, mais avec deux indicateurs mieux définis.

>Retirer les colonnes auxiliaires
Conserver ISO3, car il facilitera les futures jointures et analyses géographiques. Nous obtiendrons donc 23 colonnes dans notre jeu de référence : 22 variables analytiques et le code pays.

Pour éviter de modifier df_final avant la validation, créons df_fiabilise :

In [80]:

# Identifier les colonnes officielles auxiliaires
colonnes_auxiliaires = [
    colonne for colonne in df_final.columns
    if colonne.endswith("_WB")
    or colonne.endswith("_WUENIC")
]

# Ajouter la colonne technique de jointure
colonnes_auxiliaires.append("_merge")

# Créer le jeu fiabilisé sans les colonnes auxiliaires
df_fiabilise = df_final.drop(
    columns=colonnes_auxiliaires
).copy()

# Vérifier les dimensions
print("Dimensions :", df_fiabilise.shape)
print("Colonnes :", df_fiabilise.columns.tolist())


Dimensions : (2938, 23)
Colonnes : ['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality', 'infant deaths', 'Alcohol', 'Hepatitis B', 'Measles', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling', 'ISO3', 'Health_Expenditure_PC_USD', 'Overweight_Adults_Pct_OMS']


In [81]:
# Contrôles finaux

# Vérifier l'unicité des observations
print(
    "Doublons Country-Year :",
    df_fiabilise.duplicated(["Country", "Year"]).sum()
)

# Vérifier les codes pays
print(
    "Codes ISO3 manquants :",
    df_fiabilise["ISO3"].isna().sum()
)

# Vérifier les nouvelles variables officielles
print(
    "Dépenses de santé manquantes :",
    df_fiabilise["Health_Expenditure_PC_USD"].isna().sum()
)

print(
    "Surpoids OMS manquant :",
    df_fiabilise["Overweight_Adults_Pct_OMS"].isna().sum()
)

# Vérifier la période
print(
    "Période :",
    df_fiabilise["Year"].min(),
    "-",
    df_fiabilise["Year"].max()
)


Doublons Country-Year : 0
Codes ISO3 manquants : 0
Dépenses de santé manquantes : 92
Surpoids OMS manquant : 2
Période : 2000 - 2015


Constat : Le jeu de données fiabilisé comporte 2 938 observations et 23 variables, dont le code ISO3. Aucun doublon pays–année ni code ISO3 manquant n'a été détecté. Les indicateurs officiels des dépenses de santé par habitant et de la prévalence du surpoids présentent respectivement 92 et 2 valeurs manquantes.

Interprétation : Les contrôles confirment l'intégrité structurelle du jeu préparé. Les valeurs manquantes restantes sont documentées et ne nécessitent pas d'imputation systématique à ce stade.

Décision : Valider le jeu df_fiabilise comme base commune aux prochaines User Stories, sous réserve des vérifications spécifiques nécessaires à chaque analyse.

## 6. Prétraitements complémentaires

#### 6.1. Harmonisation des noms de colonnes

In [94]:

# Créer une copie pour l'harmonisation finale
df_fiabilise = df_fiabilise.copy()

# Définir une fonction de normalisation des noms
def nettoyer_nom_colonne(nom):

    # Convertir en minuscules et supprimer les espaces aux extrémités
    nom = nom.strip().lower()

    # Remplacer les caractères spéciaux et séparateurs par "_"
    nom = re.sub(r"[^a-z0-9]+", "_", nom)

    # Supprimer les "_" au début et à la fin
    return nom.strip("_")

# Appliquer la normalisation à toutes les colonnes
df_fiabilise.columns = [
    nettoyer_nom_colonne(col)
    for col in df_fiabilise.columns
]

# Examiner les nouveaux noms
print(df_fiabilise.columns.tolist())


['country', 'year', 'status', 'life_expectancy', 'adult_mortality', 'infant_deaths', 'alcohol', 'hepatitis_b', 'measles', 'under_five_deaths', 'polio', 'total_expenditure', 'diphtheria', 'hiv_aids', 'gdp', 'population', 'thinness_1_19_years', 'thinness_5_9_years', 'income_composition_of_resources', 'schooling', 'iso3', 'health_expenditure_pc_usd', 'overweight_adults_pct_oms']


### 6.2. Changement provisoire du nom de thinness_1_19_years

In [95]:

# Harmoniser le nom selon la description Kaggle
df_fiabilise = df_fiabilise.rename(
    columns={
        "thinness_1_19_years": "thinness_10_19_years"
    }
)

# Vérifier le nouveau nom
print(df_fiabilise.filter(like="thinness").columns.tolist())


['thinness_10_19_years', 'thinness_5_9_years']


La variable initialement nommée `thinness  1-19 years` a été renommée `thinness_10_19_years`, conformément à la tranche d'âge indiquée dans la documentation Kaggle. Cette harmonisation concerne uniquement le nom de la colonne et ne modifie aucune valeur. La correspondance exacte avec l'indicateur officiel de l'OMS reste toutefois à confirmer.

### 6.3. Harmonisation des valeurs catégorielles

In [96]:

# Supprimer les espaces superflus dans les valeurs
df_fiabilise["country"] = (
    df_fiabilise["country"].str.strip()
)

df_fiabilise["status"] = (
    df_fiabilise["status"].str.strip()
)

# Vérifier les modalités de Status
print(df_fiabilise["status"].value_counts())


status
Developing    2426
Developed      512
Name: count, dtype: int64


### 6.4. Vérification du type de year

In [97]:

# Vérifier le type de l'année et la période
print("Type :", df_fiabilise["year"].dtype)

print(
    "Période :",
    df_fiabilise["year"].min(),
    "-",
    df_fiabilise["year"].max()
)


Type : int64
Période : 2000 - 2015


### 6.5. Validation finale après harmonisation

In [107]:

# Vérifier les dimensions
print("Dimensions :", df_fiabilise.shape)

# Vérifier les noms de colonnes dupliqués
print(
    "\nNoms de colonnes dupliqués :",
    df_fiabilise.columns.duplicated().sum()
)

# Vérifier les doublons pays-année
print(
    "\nValeurs manquantes par variable :\n",
    df_fiabilise.isna().sum(),
    "\n" *2 + "Valeurs manquantes totales :",
    df_fiabilise.isna().sum().sum()
)
   

Dimensions : (2938, 23)

Noms de colonnes dupliqués : 0

Valeurs manquantes par variable :
 country                              0
year                                 0
status                               0
life_expectancy                     10
adult_mortality                     10
infant_deaths                        0
alcohol                            194
hepatitis_b                        559
measles                              0
under_five_deaths                    0
polio                               19
total_expenditure                  226
diphtheria                          19
hiv_aids                             0
gdp                                 30
population                           2
thinness_10_19_years                34
thinness_5_9_years                  34
income_composition_of_resources    167
schooling                          163
iso3                                 0
health_expenditure_pc_usd           92
overweight_adults_pct_oms            2
dtype: int6

In [108]:

# Compter les observations contenant au moins une valeur manquante
lignes_incompletes = df_fiabilise.isna().any(axis=1).sum()

# Compter les observations entièrement renseignées
lignes_completes = df_fiabilise.notna().all(axis=1).sum()

print("Observations incomplètes :", lignes_incompletes)
print("Observations complètes :", lignes_completes)


Observations incomplètes : 894
Observations complètes : 2044


**Constat :** À l'issue de la fiabilisation, le jeu de données comprend 2 044 observations entièrement renseignées (69,57 %) et 894 observations comportant au moins une valeur manquante (30,43 %). Au total, 1 561 valeurs manquantes subsistent dans les 23 colonnes.

**Interprétation :** Les valeurs manquantes sont réparties entre plusieurs variables et peuvent concerner simultanément une même observation. Leur présence ne signifie pas nécessairement que ces observations sont inutilisables pour toutes les analyses.

**Décision :** Conserver l'ensemble des 2 938 observations dans le jeu fiabilisé, sans suppression ni imputation globale. Les traitements des valeurs manquantes seront adaptés aux variables nécessaires et aux objectifs de chaque User Story.

In [109]:
#Documentation et exportation
#Nous allons maintenant enregistrer notre jeu fiabilisé dans le dossier data/processed/.


# Définir le chemin du fichier final
fichier_final = PROCESSED_DIR / "life_expectancy_fiabilise.csv"

# Exporter le jeu de données fiabilisé
df_fiabilise.to_csv(
    fichier_final,
    index=False,
    encoding="utf-8-sig"
)

# Vérifier l'enregistrement
print("Fichier enregistré :", fichier_final.exists())
print("Dimensions :", df_fiabilise.shape)




Fichier enregistré : True
Dimensions : (2938, 23)


In [110]:

# Recharger le fichier exporté
df_verification = pd.read_csv(fichier_final)

# Vérifier les dimensions et les colonnes
print("Dimensions identiques :",
      df_verification.shape == df_fiabilise.shape)

print("Colonnes identiques :",
      df_verification.columns.tolist() == df_fiabilise.columns.tolist())


Dimensions identiques : True
Colonnes identiques : True
